# Stage 0 · CPU notebook

Source readiness, verified spectral cache and explicit projection reporting. No model inference or GPU runtime is required. The GPU notebook shares the saved plans.


## 1. Storage and checkout
Mount Drive when in Colab; select the existing Stage 0 release checkout. CPU runtime is sufficient for all current cells.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

RELEASE_COMMIT = '651d9c3aa9941314374bcf1927860dd2ae2473b6'
REPOSITORY_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
# Only the dedicated Colab checkout is pinned; local work is never switched.
REPO_DIR = Path('/content/spar-stage0') if Path('/content').is_dir() else Path.cwd()
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts') if Path('/content').is_dir() else REPO_DIR / 'artifacts'
STAGE0_NAME = 'stage0-geometry-v1'
PLAN = ARTIFACT_ROOT / 'plans' / STAGE0_NAME / 'stage0.plan.json'

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
if Path('/content').is_dir():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPO_DIR)], check=True)
    changed = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True)
    if changed.strip():
        raise RuntimeError('The dedicated checkout has local changes. Preserve them before selecting a release.')
    subprocess.run(['git', 'fetch', 'origin', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
if not (REPO_DIR / 'pilot_eval').is_dir():
    raise RuntimeError('Select the repository checkout before running the workflow.')

def run_stage0(operation, *arguments):
    command = [sys.executable, '-u', '-m', 'pilot_eval', 'stage0-' + operation,
               '--output-root', str(ARTIFACT_ROOT), *map(str, arguments)]
    return subprocess.run(command, cwd=REPO_DIR, check=True)


## 2. Select and prepare
Bind the SFT/GRPO prepared sources. Optional inputs remain distinguishable from verified endpoint adapters.

In [ ]:
# Select existing prepared products; paths are relative to ARTIFACT_ROOT.
SELECTION = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'source-selection.json'
settings = {
    'sources': {
        'sft': 'plans/pilot3-inputs-v1/activation.prepared.json',
        'grpo': 'plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json',
        'random': 'plans/pilot4-grpo-fp32-v1-random-writes-v1/activation.prepared.json',
    },
    # Optional product paths must point to explicit file-hash completion receipts.
    'products': {},
}
SELECTION.parent.mkdir(parents=True, exist_ok=True)
if SELECTION.exists() and json.loads(SELECTION.read_text()) != settings:
    raise RuntimeError('Saved source selection differs. Inspect it before selecting another plan name.')
SELECTION.write_text(json.dumps(settings, indent=2) + '\n')
run_stage0('prepare', '--selection', SELECTION, '--name', STAGE0_NAME)


## 3. Audit or resume
Verify selected config, seal and adapter payloads; compare actual zero-initialization tensors. Sealed source units are reused. No optimizer-state replay, model download or inference.

In [ ]:
run_stage0('audit', '--config', PLAN)


## 4. Restore saved status
Safe in a fresh kernel after section 1. Reports audit evidence, missing prerequisites and the cumulative 20-unit budget. Unknown cost is not zero cost.

In [ ]:
run_stage0('status', '--config', PLAN)


## 5. Prepare the local spectral cache
Input: completed source audit and pinned local safetensor files. Output: an immutable cache plan. This verifies the same FP32 base hash recorded by training; it neither downloads nor constructs a model. Stage weights using the guide before this cell.

In [ ]:
AUDIT_RESULT = ARTIFACT_ROOT / 'runs' / STAGE0_NAME / 'results' / 'source-audit.json'
audit = json.loads(AUDIT_RESULT.read_text())
revision = audit['sources']['sft']['identity']['model_revision']
WEIGHTS_DIR = ARTIFACT_ROOT / 'inputs' / 'models' / revision
SPECTRAL_NAME = STAGE0_NAME + '-spectral-v1'
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / SPECTRAL_NAME / 'stage0.spectral.json'
if not (WEIGHTS_DIR / 'config.json').is_file():
    raise RuntimeError('Stage the pinned local weight snapshot at ' + str(WEIGHTS_DIR) + '; see docs/stage0-geometry.md.')
run_stage0('spectral-prepare', '--audit', PLAN, '--weights-dir', WEIGHTS_DIR, '--name', SPECTRAL_NAME)


## 6. Measure one cache unit
Output: one sealed module decomposition, measured seconds and retained bytes. This is a bounded first measurement, not a guaranteed full-run ETA. Rerunning reuses the sealed unit.

In [ ]:
run_stage0('spectral-run', '--config', SPECTRAL_PLAN, '--max-new-units', 1)


## 7. Complete or resume the cache
Decompose only missing modules. Each sealed unit retains all singular values and enough bases for k30/k50/k70 and the top-m intruder rule. Source identities, reconstruction and orthogonality are checked. No trained overlap numbers are revealed.

In [ ]:
run_stage0('spectral-run', '--config', SPECTRAL_PLAN)


## 8. Verify saved cache
Safe after restart once section 1 is run and SPECTRAL_PLAN is restored. Output: verified cache summary; later edit stages can consume these bases without opening the overlap report.

In [ ]:
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-spectral-v1') / 'stage0.spectral.json'
run_stage0('spectral-verify', '--config', SPECTRAL_PLAN)


## 9. Optional trained-overlap reveal
Leave disabled if preserving the conditional gradient-to-overlap prediction. To reveal, supply an explicit review recording either frozen-before-reveal predictions or exploratory status; the guide documents its fields. Outputs: per-module JSONL/CSV, plot and completion receipt.

In [ ]:
REVEAL_TRAINED_OVERLAPS = False
OVERLAP_REVIEW = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'overlap-review.json'
if REVEAL_TRAINED_OVERLAPS:
    run_stage0('spectral-report', '--config', SPECTRAL_PLAN, '--review', OVERLAP_REVIEW,
               '--name', STAGE0_NAME + '-principal-overlap-v1')
else:
    print('Trained overlaps remain unrevealed. Cache is ready for edit construction.')


## 10. Prepare the principal intervention catalogue
Input: verified spectral cache and audited endpoint adapters. Output: immutable edit plan. This uses the cache without opening the trained-overlap report.

In [ ]:
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-spectral-v1') / 'stage0.spectral.json'
EDIT_NAME = STAGE0_NAME + '-principal-edits-v1'
EDIT_PLAN = ARTIFACT_ROOT / 'plans' / EDIT_NAME / 'stage0.edits.json'
run_stage0('edits-prepare', '--spectral', SPECTRAL_PLAN, '--name', EDIT_NAME)


## 11. Build or resume adapters
Output: ten separate PEFT adapters, per-module check records and saved random bases. Original source adapters are never modified. Completed module units are reused after interruption. CPU only; no model is loaded.

In [ ]:
run_stage0('edits-run', '--config', EDIT_PLAN)


## 12. Verify the catalogue
Safe after restart following section 1. Output: paths, operations and verified payload identities. Construction completion does not mean GPU qualification or evaluation has passed.

In [ ]:
EDIT_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-principal-edits-v1') / 'stage0.edits.json'
run_stage0('edits-verify', '--config', EDIT_PLAN)


## 13. Restore paired collection status (CPU)
After section 1, restore the exact Ticket 05 plan produced by the GPU notebook. This checks saved batches and budget without importing a model runtime. An incomplete run stays incomplete; this cell does not regenerate outputs.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + '-loader-qualification-v1'
EVALUATION_PLAN = ARTIFACT_ROOT / 'plans' / QUALIFICATION_NAME / 'stage0.evaluation.json'
assert EVALUATION_PLAN.is_file(), 'Prepare paired collection in GPU notebook section 12 first.'
run_stage0('evaluation-status', '--config', EVALUATION_PLAN)
EVALUATION_CONFIG = json.loads(EVALUATION_PLAN.read_text())
EVALUATION_DIR = ARTIFACT_ROOT / EVALUATION_CONFIG['run_path']


## 14. Verify and inspect raw behavioral results (CPU)
Requires completed GPU collection. Verification reconciles all 1,500 responses, token/cap metadata, scorer outputs and descriptive counts against the sealed batches. The summary contains automatic scores; paired statistics and targeted answer review are later Ticket 07 work.


In [ ]:
run_stage0('evaluation-verify', '--config', EVALUATION_PLAN)
from IPython.display import Markdown, display
display(Markdown((EVALUATION_DIR / 'results' / 'results.md').read_text()))
print('Raw responses:', EVALUATION_DIR / 'results' / 'responses.jsonl')
print('Structured summary:', EVALUATION_DIR / 'results' / 'results.json')
print('Table:', EVALUATION_DIR / 'results' / 'summary.csv')


## 15. Restore fixed-context collection status (CPU)
The GPU notebook prepares and collects the KL/write product. This reads its sealed progress without running inference. Do not recover a writer while the GPU notebook is still active.


In [ ]:
DRIFT_NAME = STAGE0_NAME + '-fixed-context-v1'
DRIFT_PLAN = ARTIFACT_ROOT / 'plans' / DRIFT_NAME / 'stage0.drift.json'
assert DRIFT_PLAN.is_file(), 'Prepare fixed-context collection in GPU notebook section 16 first.'
DRIFT_CONFIG = json.loads(DRIFT_PLAN.read_text())
DRIFT_DIR = ARTIFACT_ROOT / DRIFT_CONFIG['run_path']
run_stage0('drift-status', '--config', DRIFT_PLAN)


## 16. Verify fixed-context KL and effective-write results (CPU)
Rebuilds summaries from sealed per-position KL and per-document/view/module write sums. Counted target positions are shifted for KL; write positions are unshifted. The next ticket provides paired intervals, review overlays and combined behavioral interpretation.


In [ ]:
run_stage0('drift-verify', '--config', DRIFT_PLAN)
result = json.loads((DRIFT_DIR / 'results' / 'results.json').read_text())
print('Measurement complete:', result['measurement_complete'])
print('Execution kind:', result['execution_kind'])
print('Corpus/view cells:', len(result['cells']))
for cell in result['cells']:
    print(cell['variant'], cell['corpus'], cell['view'], cell['kl'])
print('Full results:', DRIFT_DIR / 'results' / 'results.json')
print('Document sums:', DRIFT_DIR / 'results' / 'per-document.jsonl')


## 17. Restore the primary analysis plan (CPU)
Use the saved plan from GPU section 12A. If it is absent, preparation is allowed now but records that collection already started; it does not claim preregistration. This restores paths without inference.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + '-loader-qualification-v1'
EXECUTION_PLAN = ARTIFACT_ROOT / 'plans' / QUALIFICATION_NAME / 'stage0.execution.json'
EVALUATION_PLAN = EXECUTION_PLAN.parent / 'stage0.evaluation.json'
DRIFT_NAME = STAGE0_NAME + '-fixed-context-v1'
DRIFT_PLAN = ARTIFACT_ROOT / 'plans' / DRIFT_NAME / 'stage0.drift.json'
FINDINGS_NAME = STAGE0_NAME + '-primary-findings-v1'
FINDINGS_PLAN = ARTIFACT_ROOT / 'plans' / FINDINGS_NAME / 'stage0.findings.json'
run_stage0('findings-prepare', '--evaluation', EVALUATION_PLAN, '--drift', DRIFT_PLAN,
           '--name', FINDINGS_NAME)
FINDINGS_CONFIG = json.loads(FINDINGS_PLAN.read_text())
FINDINGS_DIR = ARTIFACT_ROOT / FINDINGS_CONFIG['run_path']
print('Registration:', FINDINGS_CONFIG['registration'])
print('Raw report:', FINDINGS_DIR / 'report.html')


## 18. Compute or resume paired results (CPU)
Builds all 14 declared comparisons: accuracy/strict scores, median paired length, per-document KL, module write differences, and 10,000 paired bootstrap draws with seed 42. Saves raw tables, plots and HTML. Set a unit limit to stop after a few comparisons; rerunning skips sealed ones. No human labels are assumed.


In [ ]:
FINDINGS_MAX_NEW_UNITS = None
arguments = ['--config', FINDINGS_PLAN]
if FINDINGS_MAX_NEW_UNITS is not None:
    arguments += ['--max-new-units', FINDINGS_MAX_NEW_UNITS]
run_stage0('findings-run', *arguments)
run_stage0('findings-status', '--config', FINDINGS_PLAN)
print('Inspect:', FINDINGS_DIR / 'report.html')
print('Raw table exports:', FINDINGS_DIR / 'results')


## 19. Prepare the intended-answer review (CPU)
Targets changed extracted answers, invalid endpoints and last-number diagnostic disagreement. Strict-only format failures remain automatic tables. Identical prompt/response text is deduplicated. Optional reuse requires a verified historical audit configuration and explicit revision; no fuzzy text matching. The page hides source identity, gold and scorer results, but separate files are not an access-control boundary.


In [ ]:
REUSE_CONFIRMED_AUDIT = ''
REUSE_CONFIRMED_REVISION = ''
arguments = ['--config', FINDINGS_PLAN]
if REUSE_CONFIRMED_AUDIT.strip():
    arguments += ['--reuse-audit', REUSE_CONFIRMED_AUDIT]
    if REUSE_CONFIRMED_REVISION.strip():
        arguments += ['--reuse-revision', REUSE_CONFIRMED_REVISION]
run_stage0('findings-audit-prepare', *arguments)
print('Open the reviewer page:', FINDINGS_DIR / 'audit' / 'reviewer' / 'review.html')
print('After saving labels, import the updated draft before continuing:',
      FINDINGS_DIR / 'audit' / 'reviewer' / 'draft.json')


## 20. Save an exported review draft (CPU; repeat as needed)
Enter the path to the downloaded draft. Numeric labels require a number, not a correctness decision. Saving merges with the durable revision; a stale draft is rejected. Incomplete review does not block a raw-results report.


In [ ]:
REVIEW_DRAFT_PATH = ''
if not REVIEW_DRAFT_PATH.strip():
    raise ValueError('Enter the path to the downloaded reviewer JSON draft.')
run_stage0('findings-audit-save', '--config', FINDINGS_PLAN, '--labels', REVIEW_DRAFT_PATH)
run_stage0('findings-audit-status', '--config', FINDINGS_PLAN)


## 21. Freeze and reveal completed review (CPU; only after all answers are labelled)
Freezes the identified answers before revealing gold/scorer comparisons. Caps keep their original zero-reward rule. Confirmed numeric cases form a separately labelled overlay; absent/ambiguous labels are retained without inventing a numeric correction. Corrections after freeze require a new named analysis/audit with exposed timing, preserving this one.


In [ ]:
FREEZE_COMPLETED_REVIEW = False
if FREEZE_COMPLETED_REVIEW:
    run_stage0('findings-audit-freeze', '--config', FINDINGS_PLAN)
    run_stage0('findings-audit-reveal', '--config', FINDINGS_PLAN)
else:
    run_stage0('findings-audit-status', '--config', FINDINGS_PLAN)


## 22. Verify and publish the report snapshot (CPU)
Reconstructs raw estimates and intervals against saved pairs and draws. Publishes a new immutable HTML/table snapshot with current review status; the original raw report remains intact. Pending review and absent optional experiments do not prevent publication. These files do not establish real execution when their source is a synthetic fixture.


In [ ]:
run_stage0('findings-verify', '--config', FINDINGS_PLAN)
run_stage0('findings-report', '--config', FINDINGS_PLAN)
print('Original raw report:', FINDINGS_DIR / 'report.html')
print('Review-status report snapshots:', FINDINGS_DIR / 'reports')
print('JSON / JSONL / CSV:', FINDINGS_DIR / 'results')
print('Exportable SVG / PNG figures:', FINDINGS_DIR / 'plots')
